# Représenter les expéditions de pèche à la baleine avec pandas et folium

## Exploration du jeu de données

In [24]:
import pandas as pd
import folium as f
import numpy as np

In [25]:
df=pd.read_csv("data/AmericanOffshoreWhalingLogbookData/aowl.csv", sep=';')

C:\Users\tenep\AppData\Local\Temp\ipykernel_23080\499177789.py:1: DtypeWarning: Columns (0: Remarks) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("data/AmericanOffshoreWhalingLogbookData/aowl.csv", sep=';')


In [26]:
df.shape

(468249, 14)

In [27]:
df.dtypes

sequence       int64
VoyageID         str
Lat          float64
Lon          float64
Day            int64
Month          int64
Year           int64
Encounter        str
Species          str
NStruck      float64
NTried       float64
Place            str
Source           str
Remarks          str
dtype: object

In [28]:
df.isna().sum()

sequence          0
VoyageID          0
Lat               0
Lon               0
Day               0
Month             0
Year              0
Encounter         0
Species      391966
NStruck      392314
NTried       392323
Place        450360
Source            0
Remarks      466690
dtype: int64

In [29]:
df.head(20)

,sequence,VoyageID,Lat,Lon,Day,Month,Year,Encounter,Species,NStruck,NTried,Place,Source,Remarks
0,1,AV00272,41.64,-70.93,9,8,1875,NoEnc,NaN,NaN,NaN,New Bedford,CoML,NaN
1,2,AV00272,40.75,-70.75,10,8,1875,NoEnc,NaN,NaN,NaN,NaN,CoML,NaN
2,3,AV00272,39.66,-70.33,11,8,1875,NoEnc,NaN,NaN,NaN,NaN,CoML,NaN
3,4,AV00272,39.26,-69.41,12,8,1875,NoEnc,NaN,NaN,NaN,NaN,CoML,NaN
4,5,AV00272,38.80,-67.83,13,8,1875,NoEnc,NaN,NaN,NaN,NaN,CoML,NaN
5,6,AV00272,37.63,-63.66,14,8,1875,NoEnc,NaN,NaN,NaN,NaN,CoML,NaN
6,7,AV00272,35.46,-60.75,15,8,1875,Sight,Sperm,0.0,0.0,NaN,CoML,NaN
7,8,AV00272,35.96,-60.08,16,8,1875,NoEnc,NaN,NaN,NaN,NaN,CoML,NaN
8,9,AV00272,36.06,-57.50,17,8,1875,NoEnc,NaN,NaN,NaN,NaN,CoML,NaN
9,10,AV00272,35.58,-55.41,18,8,1875,NoEnc,NaN,NaN,NaN,NaN,CoML,NaN


In [30]:
df['Date_str'] = (df['Day'].astype(str).str.zfill(2) + '/' + 
                  df['Month'].astype(str).str.zfill(2) + '/' + 
                  df['Year'].astype(str))
df['Date'] = pd.to_datetime(df['Date_str'], format='%d/%m/%Y', errors='coerce')

In [31]:
df['Date']

0        1875-08-09
1        1875-08-10
2        1875-08-11
3        1875-08-12
4        1875-08-13
            ...    
468244   1857-02-17
468245   1857-02-22
468246   1857-03-04
468247   1857-03-08
468248   1858-02-21
Name: Date, Length: 468249, dtype: datetime64[us]

In [32]:
nb_voyage=df["VoyageID"].unique().__len__()
print(f"Il y a {nb_voyage} voyages différents répertoriés")

Il y a 1449 voyages différents répertoriés


In [33]:
df["Species"].value_counts()

Species
Sperm        24473
Whale        21343
Right        16746
Finback       3888
Pilot         2750
Bowhead       2485
Humpback      2376
Grampus       1029
Gray           612
Killer         305
Blue           230
Dolphin         19
Blackfish       14
Porpoise        12
Killers          1
Name: count, dtype: int64

In [34]:
nb_espèces=df["Species"].value_counts().index
print("Les espèces chassées sont les suivantes : ")
for i in nb_espèces:
    print(f"- {i}")

Les espèces chassées sont les suivantes : 
- Sperm
- Whale
- Right
- Finback
- Pilot
- Bowhead
- Humpback
- Grampus
- Gray
- Killer
- Blue
- Dolphin
- Blackfish
- Porpoise
- Killers


In [35]:
nb_strike=df[(df["Species"].notna()) & (df["Encounter"]=="Strike")]
nb_strike["Species"].value_counts()

Species
Sperm        15646
Right         7889
Whale         2558
Bowhead       2047
Humpback      1186
Pilot          703
Gray           358
Killer          28
Blue            20
Finback         15
Grampus         13
Dolphin          4
Blackfish        1
Porpoise         1
Name: count, dtype: int64

In [36]:
print(f"{nb_strike["Species"].value_counts().sum()} baleines ont été touchées par un harpon")

30469 baleines ont été touchées par un harpon


## Fonctions

In [37]:
def newMap(lat,lon,zoom_start=6):
    m=f.Map(location=[lat,lon],zoom_start=zoom_start)
    return m

In [38]:
def addMarker(radius,data,m):
    for lat,lon,enc,date,whale in zip(data['Lat'], data['LonNorm'], data['Encounter'],data['Date'], data['Species']):
        if enc=="Spoke":
            color='green'
        elif enc=="Sight":
            color="yellow"
        elif enc=="Strike":
            color="red"
        else :
            color=''
        f.CircleMarker(
            location=[lat, lon],
            radius=radius,
            color="black",
            weight=1,
            opacity=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.6,
            popup="{} - {} - {}".format(date, enc, whale),
        ).add_to(m)

In [39]:
def addLine(data,m):
    coordinates=[]
    for lat,lon in zip(data['Lat'], data['LonNorm']):
        coordinates.append([lat,lon])
    f.PolyLine(
    locations=coordinates,
    line_cap="butt",
    color="#F00EB7",
    weight=2,
).add_to(m)

In [40]:
def dfVoyage(df):
    ligneVoyage=df['VoyageID'].sample(n=1).index[0]
    voyage=df['VoyageID'].loc[ligneVoyage]
    df_voyage=df[df['VoyageID']==voyage]
    df_voyage=df_voyage.sort_values('Date')
    return df_voyage

In [41]:
def lonUnwrap(df_voyage):
    """Déplie la longitude pour éviter les sauts, sans la contraindre à -180/+180"""
    lons = df_voyage['Lon'].values.copy()
    lons_unwrapped = np.degrees(np.unwrap(np.radians(lons)))
    df_voyage = df_voyage.copy()
    df_voyage['LonNorm'] = lons_unwrapped
    return df_voyage

## Représentation d'un voyage

Voyage aléatoire

In [ ]:
voyage=dfVoyage(df)
voyage = lonUnwrap(voyage) 
m=newMap(0,0,2)
addMarker(10,voyage,m)
addLine(voyage,m)
m.save("maps/map_alea.html")
print(voyage['VoyageID'].head(1))

241716    AV10432
Name: VoyageID, dtype: str


Voyage choisi

In [ ]:
voyage=df.loc[df['VoyageID']=='AV03829']
voyage=voyage.sort_values('Date')
voyage = lonUnwrap(voyage) 
m=newMap(0,0,2)
addMarker(10,voyage,m)
addLine(voyage,m)
m.save("maps/map_choisi.html")

Sur certains voyages il n'y a pas de points d'arriver ou de départ via un port ou une zone sur terre. Après investigation c'est un manque de données.

## Là où les baleines ont été vues et/ou chassées

In [21]:
df_sight=df[df['Encounter']=='Sight']
df_sight

,sequence,VoyageID,Lat,Lon,Day,Month,Year,Encounter,Species,NStruck,NTried,Place,Source,Remarks,Date_str,Date
6,7,AV00272,35.460,-60.750,15,8,1875,Sight,Sperm,0.0,0.0,NaN,CoML,NaN,15/08/1875,1875-08-15
23,24,AV00272,35.560,-49.710,1,9,1875,Sight,Sperm,0.0,0.0,NaN,CoML,NaN,01/09/1875,1875-09-01
25,26,AV00272,35.180,-49.620,2,9,1875,Sight,Sperm,0.0,0.0,NaN,CoML,NaN,02/09/1875,1875-09-02
26,27,AV00272,35.530,-49.530,3,9,1875,Sight,Sperm,0.0,0.0,NaN,CoML,NaN,03/09/1875,1875-09-03
29,30,AV00272,36.430,-50.110,6,9,1875,Sight,Sperm,0.0,0.0,NaN,CoML,NaN,06/09/1875,1875-09-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
467035,4000992,BV049510,-16.758,11.745,25,7,1801,Sight,Right,0.0,0.0,NaN,BSWF Log,Could not strike,25/07/1801,1801-07-25
467042,4000999,BV049510,-16.750,11.723,11,8,1801,Sight,Humpback,0.0,0.0,NaN,BSWF Log,Chased a humpback,11/08/1801,1801-08-11
467043,4001000,BV049510,-16.497,11.765,17,8,1801,Sight,Right,0.0,0.0,NaN,BSWF Log,Could not get up with the whale,17/08/1801,1801-08-17
467049,4001006,BV049510,-16.671,11.781,7,9,1801,Sight,Right,0.0,0.0,NaN,BSWF Log,Saw one whale,07/09/1801,1801-09-07


In [ ]:
m=newMap(0,0,2)
df_sight = lonUnwrap(df_sight) 
addMarker(10,df_sight,m)
m.save("maps/map_sight.html")